# Strong Models Comparison for PM2.5 Prediction

This notebook extends the previous regression workflow with stronger tabular candidates available in the current scikit-learn stack.

Models compared here:
- Linear Regression
- Random Forest
- Gradient Boosting
- Extra Trees
- Support Vector Regression (RBF)
- Neural Network (MLPRegressor)

The notebook ends with one consolidated comparison across all models on the hold-out test set.

In [ ]:
import os
import sys
import json
import warnings
from pathlib import Path

import altair as alt
import pandas as pd
from sklearn.ensemble import ExtraTreesRegressor, GradientBoostingRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, RandomizedSearchCV, cross_validate, train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

warnings.filterwarnings("ignore")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")
alt.data_transformers.disable_max_rows()
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.3f}")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.processing.preprocess import build_preprocessor_from_settings, load_raw_dataframe, preprocess_dataframe
from src.utils.config import get_nested, load_settings, resolve_path

def build_pipeline(estimator):
    return Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", estimator),
        ]
    )

def compute_metrics(y_true, y_pred):
    return {
        "rmse": float(mean_squared_error(y_true, y_pred) ** 0.5),
        "mae": float(mean_absolute_error(y_true, y_pred)),
        "r2": float(r2_score(y_true, y_pred)),
    }


## 1. Load data

We keep the same target (`pm25`) and the same core predictors already used in the project pipeline, so the comparison stays aligned with the rest of the application.

In [ ]:
settings = load_settings(PROJECT_ROOT / "config/settings.yaml")
preprocessor = build_preprocessor_from_settings(settings)
raw_df = load_raw_dataframe(PROJECT_ROOT / "data/raw/air_quality_combined_source.csv")
df = preprocess_dataframe(raw_df, preprocessor).copy()

feature_columns = get_nested(settings, "ml", "feature_columns", default=[])
target_column = get_nested(settings, "ml", "target_column", default="pm25")

model_df = df[feature_columns + [target_column]].dropna(subset=[target_column]).copy()
X = model_df[feature_columns]
y = model_df[target_column]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

summary_df = pd.DataFrame(
    {
        "metric": ["rows", "train_rows", "test_rows", "n_features"],
        "value": [len(model_df), len(X_train), len(X_test), len(feature_columns)],
    }
)
display(summary_df)
display(model_df.describe().T)


## 2. Baseline cross-validation across all models

This first comparison uses the default or near-default versions of the six models. It tells us which families are promising **before** any tuning.

In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    "rmse": "neg_root_mean_squared_error",
    "mae": "neg_mean_absolute_error",
    "r2": "r2",
}

baseline_models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(random_state=42, n_jobs=1),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(random_state=42, n_jobs=1),
    "SVR (RBF)": SVR(kernel="rbf", C=10.0, epsilon=0.2),
    "Neural Network (MLP)": MLPRegressor(
        random_state=42,
        max_iter=1200,
        early_stopping=True,
        hidden_layer_sizes=(64, 32),
    ),
}

baseline_rows = []
for model_name, estimator in baseline_models.items():
    scores = cross_validate(build_pipeline(estimator), X_train, y_train, cv=cv, scoring=scoring, n_jobs=1)
    baseline_rows.append(
        {
            "model": model_name,
            "cv_rmse": -scores["test_rmse"].mean(),
            "cv_mae": -scores["test_mae"].mean(),
            "cv_r2": scores["test_r2"].mean(),
        }
    )

baseline_results = pd.DataFrame(baseline_rows).sort_values("cv_rmse").reset_index(drop=True)
display(baseline_results)
print(
    f"Best baseline by CV RMSE: {baseline_results.iloc[0]['model']} "
    f"({baseline_results.iloc[0]['cv_rmse']:.3f})"
)


## 3. Light tuning of the strongest non-linear models

To keep the notebook rerunnable, the search is intentionally light (`n_iter=3`). The idea is not exhaustive optimization but a fair test of whether each model family can improve with a small hyperparameter search.

In [ ]:
search_spaces = {
    "Random Forest tuned": (
        RandomForestRegressor(random_state=42, n_jobs=1),
        {
            "model__n_estimators": [150, 250],
            "model__max_depth": [None, 8, 12],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [1.0, "sqrt"],
        },
    ),
    "Gradient Boosting tuned": (
        GradientBoostingRegressor(random_state=42),
        {
            "model__n_estimators": [150, 250],
            "model__learning_rate": [0.03, 0.05, 0.08],
            "model__max_depth": [2, 3, 4],
            "model__subsample": [0.8, 1.0],
        },
    ),
    "Extra Trees tuned": (
        ExtraTreesRegressor(random_state=42, n_jobs=1),
        {
            "model__n_estimators": [150, 250],
            "model__max_depth": [None, 8, 12],
            "model__min_samples_leaf": [1, 2, 4],
            "model__max_features": [1.0, "sqrt"],
        },
    ),
    "SVR (RBF) tuned": (
        SVR(kernel="rbf"),
        {
            "model__C": [1.0, 10.0, 25.0, 50.0],
            "model__epsilon": [0.1, 0.2, 0.5, 1.0],
            "model__gamma": ["scale", 0.05, 0.1],
        },
    ),
    "Neural Network (MLP) tuned": (
        MLPRegressor(random_state=42, max_iter=1400, early_stopping=True),
        {
            "model__hidden_layer_sizes": [(64,), (128,), (64, 32)],
            "model__alpha": [0.0001, 0.001, 0.01],
            "model__learning_rate_init": [0.001, 0.005, 0.01],
        },
    ),
}

tuned_models = {}
tuning_rows = []
for model_name, (estimator, params) in search_spaces.items():
    search = RandomizedSearchCV(
        estimator=build_pipeline(estimator),
        param_distributions=params,
        n_iter=3,
        scoring="neg_root_mean_squared_error",
        cv=cv,
        n_jobs=1,
        random_state=42,
    )
    search.fit(X_train, y_train)
    tuned_models[model_name] = search.best_estimator_
    tuning_rows.append(
        {
            "model": model_name,
            "best_cv_rmse": -search.best_score_,
            "best_params": json.dumps(search.best_params_),
        }
    )

tuning_results = pd.DataFrame(tuning_rows).sort_values("best_cv_rmse").reset_index(drop=True)
display(tuning_results)


## 4. Final hold-out comparison between all models

This table is the final answer of the notebook. Every model is evaluated on the same unseen test set, and the ranking is sorted by test RMSE.

In [ ]:
final_models = {"Linear Regression": build_pipeline(LinearRegression())}
final_models.update(tuned_models)

final_rows = []
prediction_store = {}
for model_name, model in final_models.items():
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    prediction_store[model_name] = predictions
    metrics = compute_metrics(y_test, predictions)
    final_rows.append({"model": model_name, **metrics})

final_results = pd.DataFrame(final_rows).sort_values("rmse").reset_index(drop=True)
display(final_results.rename(columns={"rmse": "test_rmse", "mae": "test_mae", "r2": "test_r2"}))

best_model_name = final_results.iloc[0]["model"]
print(f"Best final model on the hold-out test set: {best_model_name}")

summary_payload = {
    "baseline_results": baseline_results.to_dict(orient="records"),
    "tuning_results": tuning_results.to_dict(orient="records"),
    "final_results": final_results.to_dict(orient="records"),
    "best_model": best_model_name,
}
summary_path = resolve_path("models/strong_models_comparison_summary.json")
summary_path.write_text(json.dumps(summary_payload, indent=2), encoding="utf-8")


## 5. Actual values vs predictions

The chart below overlays the hold-out test predictions of all final models against the real PM2.5 values.

In [ ]:
prediction_frame = pd.DataFrame(
    {
        "sample_index": list(range(len(y_test))),
        "actual_pm25": y_test.reset_index(drop=True),
    }
)

for model_name, predictions in prediction_store.items():
    safe_name = model_name.lower().replace(" ", "_").replace("(", "").replace(")", "")
    prediction_frame[f"prediction_{safe_name}"] = predictions

long_plot_df = prediction_frame.melt(
    id_vars="sample_index",
    value_vars=[column for column in prediction_frame.columns if column != "sample_index"],
    var_name="series",
    value_name="value",
)

label_map = {
    "actual_pm25": "Actual PM2.5",
    "prediction_linear_regression": "Prediction - Linear Regression",
    "prediction_random_forest_tuned": "Prediction - Random Forest tuned",
    "prediction_gradient_boosting_tuned": "Prediction - Gradient Boosting tuned",
    "prediction_extra_trees_tuned": "Prediction - Extra Trees tuned",
    "prediction_svr_rbf_tuned": "Prediction - SVR (RBF) tuned",
    "prediction_neural_network_mlp_tuned": "Prediction - Neural Network (MLP) tuned",
}
long_plot_df["series"] = long_plot_df["series"].map(label_map).fillna(long_plot_df["series"])

chart = (
    alt.Chart(long_plot_df)
    .mark_line(strokeWidth=2.2)
    .encode(
        x=alt.X("sample_index:Q", title="Test sample"),
        y=alt.Y("value:Q", title="PM2.5"),
        color=alt.Color("series:N", title="Series"),
        tooltip=[
            alt.Tooltip("sample_index:Q", title="Test sample"),
            alt.Tooltip("series:N", title="Series"),
            alt.Tooltip("value:Q", title="Value", format=".2f"),
        ],
    )
    .properties(height=420, title="Actual PM2.5 vs predictions across all final models")
    .interactive()
)

plot_path = resolve_path("models/strong_models_actual_vs_predicted.html")
csv_path = resolve_path("models/strong_models_actual_vs_predicted.csv")
plot_path.parent.mkdir(parents=True, exist_ok=True)
prediction_frame.to_csv(csv_path, index=False)
chart.save(plot_path)

display(chart)
print(f"Saved comparison plot to: {plot_path}")
print(f"Saved prediction table to: {csv_path}")


## 6. Final interpretation

This last cell prints a short report-ready conclusion based on the final hold-out ranking.

In [ ]:
best_row = final_results.iloc[0]
worst_row = final_results.iloc[-1]

conclusion = (
    f"Final conclusion: the best model in this expanded comparison is {best_row['model']} "
    f"with RMSE={best_row['rmse']:.3f}, MAE={best_row['mae']:.3f}, and R2={best_row['r2']:.3f} on the hold-out test set. "
    f"The weakest final model is {worst_row['model']} with RMSE={worst_row['rmse']:.3f}. "
    "This ranking should be preferred over training-only impressions because every model here is compared on the same unseen data."
)

print(conclusion)
print("\nArtifacts:")
print(f"- {summary_path}")
print(f"- {plot_path}")
print(f"- {csv_path}")
